# USE THIS WITH DEFAULT ADB DATA

### Load the saved model

In [ ]:
import pickle
import pyarrow.dataset as ds
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import GroupShuffleSplit
import pickle

short_names = {
    'Pòlissa/Póliza/Policy': 'POLICY',
    'Tecnologia/Tecnología/Technology': 'TECHNOLOGY',
    'Diàmetre comptador (cm)/Diámetro contador (cm)/Counter diameter (cm)': 'DIAMETER',
    'Ús/Uso/Use': 'USAGE',
    "Tipus d'habitatge/Tipo de vivienda/Type of housing": 'HOUSING',
    'Data/Fecha/Date': 'DATETIME',
    'Índex de lectura (L/h)/Índice de lectura (L/h)/Reading index (L/h)': 'CONSUMPTION',
}

WINDOW_SIZE = 4
model = pickle.load(open("../data/models/model_test.sav", 'rb'))

In [ ]:
import pandas as pd

def setup_input(df, window_size = 4, output_path=None):
    """
    Optimized function to create a sliding window dataset.
    
    Args:
    - df (pd.DataFrame): Input DataFrame.
    - window_size (int): Size of the sliding window.
    - output_path (str): Path to save the output. If None, returns a DataFrame.
    
    Returns:
    - pd.DataFrame or None: Processed DataFrame if `output_path` is None.
    """
    # Create a generator to yield rows for each window
    def generate_rows():
        for policy, group in df.groupby('POLICY'):
            # Convert columns to NumPy arrays for efficient slicing
            flows = group['FLOW'].to_numpy()
            technology = group['TECHNOLOGY'].to_numpy()
            usage = group['USAGE'].to_numpy()
            housing = group['HOUSING'].to_numpy()
            consumption = group['CONSUMPTION'].to_numpy()
            hour_date = group['DATETIME'].to_numpy()
            weekday = group['WEEKDAY'].to_numpy()
            hours = group['HOUR'].to_numpy()

            # Generate sliding windows
            for i in range(len(flows) - window_size + 1):
                row = {
                    'POLICY': policy,
                    'TECHNOLOGY': technology[i],
                    'USAGE': usage[i],
                    'HOUSING': housing[i],
                    'CONSUMPTION': consumption[i],
                    'DATETIME': hour_date[i],
                    'WEEKDAY': weekday[i],
                    'START_HOUR': hours[i],
                }
                # Add flow values for the window
                for j in range(window_size):
                    row[f'FLOW_{j + 1}'] = flows[i + j]
                yield row

    # Write to file or return as DataFrame
    if output_path:
        pd.DataFrame.from_records(generate_rows()).to_parquet(output_path, index=False)
        print(f"Saved processed data to {output_path}")
        return None
    else:
        return pd.DataFrame.from_records(generate_rows())


### Load and format the data

In [ ]:
file_path = '../data/splits/split_29.parquet'

dataset = ds.dataset(file_path, format="parquet")
table = dataset.to_table()
df = table.to_pandas()
df = df.rename(columns=short_names)

df['DATETIME'] = pd.to_datetime(df['DATETIME'])
df['WEEKDAY'] = df['DATETIME'].dt.dayofweek  # Extract weekday (Monday=0, Sunday=6)
df['HOUR'] = df['DATETIME'].dt.hour

df['FLOW'] = df.groupby('POLICY')['CONSUMPTION'].diff()
df = df.dropna(subset=["FLOW"])

df_set = setup_input(df)

# Dynamically construct the FLOW column names
flow_columns = [f"FLOW_{i}" for i in range(1, WINDOW_SIZE + 1)]
# Define the full feature list
base_features = ['USAGE', 'HOUSING', 'WEEKDAY', 'START_HOUR']
features = base_features + flow_columns

# Select features and target
X = df_set[features]
label_encoders = {}

for col in X.select_dtypes(include='object').columns:  # Select categorical columns
    le = LabelEncoder()
    X.loc[:, col] = le.fit_transform(X[col])  # Use .loc[] for explicit assignment
    label_encoders[col] = le

In [ ]:
y = model.predict(X)

In [ ]:
df_set['LEAK'] = y

In [ ]:
counts = df_set['LEAK'].value_counts()
print(counts)

In [ ]:
df_set[df_set["LEAK"] == True][0:50]

In [ ]:
# Ensure the data is sorted by POLICY and DATETIME for proper processing
df_set['DATETIME'] = pd.to_datetime(df_set['DATETIME'])
df_set = df_set.sort_values(by=['POLICY', 'DATETIME'])

# Group by POLICY and detect consecutive leaks
df_set['LEAK_GROUP'] = (df_set['LEAK'] != df_set['LEAK'].shift()).cumsum()  # Identify groups of leaks
leak_groups = df_set[df_set['LEAK'] == 1].groupby(['POLICY', 'LEAK_GROUP']).agg({
    'DATETIME': ['min', 'max'],  # Leak start and end
    'LEAK': 'size'  # Leak duration
}).reset_index()

# Flatten the multi-level columns after aggregation
leak_groups.columns = ['POLICY', 'LEAK_GROUP', 'START_DATETIME', 'END_DATETIME', 'DURATION']
leak_groups['DURATION'] += 3

# Create a summary of leaks by duration
leak_summary = leak_groups.groupby('DURATION').size().reset_index(name='NUM_LEAKS')

print(leak_summary)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
leak_summary['NORM'] = leak_summary['DURATION'] ** 0.1

plt.style.use('dark_background')
# Use palette and assign hue to the x-axis variable
plt.figure(figsize=(10, 6), facecolor='#0e1117')
sns.barplot(x='DURATION', y='NUM_LEAKS', data=leak_summary, hue='NORM', palette='Blues_d', dodge=False, legend=False)
plt.xlabel('Leak Duration (Number of Rows)', fontsize=12)
plt.ylabel('Number of Leaks', fontsize=12)
plt.title('Histogram of Leak Durations', fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()



In [ ]:
# Extract leak start hours from LEAK_GROUP data
leak_start_hours = df_set[df_set['LEAK'] == 1].groupby(['START_HOUR']).size().reset_index(name='NUM_LEAKS')

# Plot: Bar chart of leaks by start hour
plt.figure(figsize=(10, 6), facecolor='#0e1117')
sns.barplot(x='START_HOUR', y='NUM_LEAKS', data=leak_start_hours, color='skyblue', edgecolor='black')
plt.xlabel('Start Hour', fontsize=12)
plt.ylabel('Number of Leaks', fontsize=12)
plt.title('Number of Leaks by Start Hour', fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


In [ ]:
# Group the dataset by USAGE and HOUSING and count the number of leaks
leak_start_hours = df_set[df_set['LEAK'] == 1].groupby(['USAGE', 'HOUSING']).size().reset_index(name='NUM_LEAKS')

# Combine USAGE and HOUSING into a single column for visualization
leak_start_hours['USAGE_HOUSING'] = leak_start_hours['USAGE'] + ' - ' + leak_start_hours['HOUSING']

# Plot: Bar chart with combined USAGE and HOUSING
plt.figure(figsize=(12, 6))
sns.barplot(x='USAGE_HOUSING', y='NUM_LEAKS', data=leak_start_hours, color='skyblue', edgecolor='black')
plt.xlabel('Usage - Housing', fontsize=12)
plt.ylabel('Number of Leaks', fontsize=12)
plt.title('Number of Leaks by Usage and Housing', fontsize=14)
plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels for better readability
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


In [ ]:
# Extract leak start hours from LEAK_GROUP data (unchanged)
leak_start_hours = df_set[df_set['LEAK'] == 1].groupby(['USAGE']).size().reset_index(name='NUM_LEAKS')

# Define a colormap
cmap = plt.cm.Blues

# Limit colormap range for smoother gradient (adjust start and end values)
start_color = 0.8  # Adjust starting point in the colormap (0 to 1)
end_color = 0.2  # Adjust ending point in the colormap (0 to 1)

# Plot: Pie chart of leaks by start hour
plt.figure(figsize=(8, 8))
plt.pie(leak_start_hours['NUM_LEAKS'], labels=leak_start_hours['USAGE'], autopct="%1.1f%%", startangle=140, colors=cmap(np.linspace(start_color, end_color, len(leak_start_hours))))
plt.axis('equal')
plt.title('Number of Leaks by Usage', fontsize=14)
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.show()

In [ ]:
# Extract leak start hours from LEAK_GROUP data (unchanged)
leak_start_hours = df_set[df_set['LEAK'] == 1].groupby(['HOUSING']).size().reset_index(name='NUM_LEAKS')

# Define a colormap
cmap = plt.cm.Blues

# Limit colormap range for smoother gradient (adjust start and end values)
start_color = 0.8  # Adjust starting point in the colormap (0 to 1)
end_color = 0.2  # Adjust ending point in the colormap (0 to 1)

# Plot: Pie chart of leaks by start hour
plt.figure(figsize=(8, 8))
plt.pie(leak_start_hours['NUM_LEAKS'], labels=leak_start_hours['HOUSING'], autopct="%1.1f%%", startangle=140, colors=cmap(np.linspace(start_color, end_color, len(leak_start_hours))))
plt.axis('equal')
plt.title('Number of Leaks by Usage', fontsize=14)
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.show()

In [ ]:
import streamlit as st 
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

def plot_durations(df_set):
    # Ensure the data is sorted by POLICY and DATETIME for proper processing
    df_set['DATETIME'] = pd.to_datetime(df_set['DATETIME'])
    df_set = df_set.sort_values(by=['POLICY', 'DATETIME'])

    # Group by POLICY and detect consecutive leaks
    df_set['LEAK_GROUP'] = (df_set['LEAK'] != df_set['LEAK'].shift()).cumsum()  # Identify groups of leaks
    leak_groups = df_set[df_set['LEAK'] == 1].groupby(['POLICY', 'LEAK_GROUP']).agg({
        'DATETIME': ['min', 'max'],  # Leak start and end
        'LEAK': 'size'  # Leak duration
    }).reset_index()

    # Flatten the multi-level columns after aggregation
    leak_groups.columns = ['POLICY', 'LEAK_GROUP', 'START_DATETIME', 'END_DATETIME', 'DURATION']
    leak_groups['DURATION'] += 3

    # Create a summary of leaks by duration
    leak_summary = leak_groups.groupby('DURATION').size().reset_index(name='NUM_LEAKS')
    leak_summary['NORM'] = leak_summary['DURATION'] ** 0.1

    # Use dark theme from matplotlib
    plt.style.use('dark_background')

    # Create a figure for multiple subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 16), facecolor='#0e1117')  # Adjust the size for more plots
    axes = axes.flatten()  # Flatten axes array for easy indexing



    # Plot 1: Histogram of Leak Durations
    sns.barplot(x='DURATION', y='NUM_LEAKS', data=leak_summary, hue='NORM', palette='Blues_d', dodge=False, legend=False, ax=axes[0])
    axes[0].set_xlabel('Leak Duration (Number of Rows)', fontsize=12)
    axes[0].set_ylabel('Number of Leaks', fontsize=12)
    axes[0].set_title('Histogram of Leak Durations', fontsize=14)
    axes[0].grid(axis='y', linestyle='--', alpha=0.7)

    # Plot 2: Bar chart of leaks by start hour
    leak_start_hours = df_set[df_set['LEAK'] == 1].groupby(['START_HOUR']).size().reset_index(name='NUM_LEAKS')
    sns.barplot(x='START_HOUR', y='NUM_LEAKS', data=leak_start_hours, color='skyblue', edgecolor='black', ax=axes[1])
    axes[1].set_xlabel('Start Hour', fontsize=12)
    axes[1].set_ylabel('Number of Leaks', fontsize=12)
    axes[1].set_title('Number of Leaks by Start Hour', fontsize=14)
    axes[1].grid(axis='y', linestyle='--', alpha=0.7)

    # Plot 3: Pie chart of leaks by Usage
    leak_usage = df_set[df_set['LEAK'] == 1].groupby(['USAGE']).size().reset_index(name='NUM_LEAKS')
    cmap = plt.cm.Blues
    start_color = 0.9
    end_color = 0.5
    axes[2].pie(leak_usage['NUM_LEAKS'], labels=leak_usage['USAGE'], autopct="%1.1f%%", startangle=140, colors=cmap(np.linspace(start_color, end_color, len(leak_usage))))
    axes[2].axis('equal')
    axes[2].set_title('Number of Leaks by Usage', fontsize=14)

    # Plot 4: Pie chart of leaks by Housing
    leak_housing = df_set[df_set['LEAK'] == 1].groupby(['HOUSING']).size().reset_index(name='NUM_LEAKS')
    axes[3].pie(leak_housing['NUM_LEAKS'], labels=leak_housing['HOUSING'], autopct="%1.1f%%", startangle=140, colors=cmap(np.linspace(start_color, end_color, len(leak_housing))))
    axes[3].axis('equal')
    axes[3].set_title('Number of Leaks by Housing', fontsize=14)

    # Display the figure in Streamlit
    plt.plot()


plot_durations(df_set)
